# Notebook 06: Year-over-Year Comparison

**One Sensor, One Year — Edition 1: India Breathes**

Is 2024 different from 2023 and 2022? Is solar really growing that fast? Is coal actually declining? Let's put three years side by side.

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv('../data/processed/india_all_years.csv', parse_dates=['date'])

# Filter to 2022-2024
df = df[(df['date'] >= '2022-01-01') & (df['date'] <= '2024-12-31')].copy()
df['year'] = df['date'].dt.year
df['day_of_year'] = df['date'].dt.dayofyear

# Compute total (same as notebook 01)
# total_ex_re + renewables = total
if 'total' not in df.columns:
    df['total'] = df['total_ex_re'] + df['renewables']

# Clean share
df['clean_share'] = ((df['nuclear'] + df['hydro'] + df['renewables']) / df['total'] * 100)
df['re_share'] = (df['renewables'] / df['total'] * 100)
df['wind_solar'] = df['wind'].fillna(0) + df['solar'].fillna(0)

years = [2022, 2023, 2024]
year_colors = {2022: '#3498DB', 2023: '#E67E22', 2024: '#2ECC71'}

print(f'Data loaded: {len(df)} days across {df["year"].nunique()} years')
for y in years:
    n = len(df[df['year'] == y])
    print(f'  {y}: {n} days')

Data loaded: 1096 days across 3 years
  2022: 365 days
  2023: 365 days
  2024: 366 days


## 1. Annual Totals — The Growth Story

In [2]:
fuel_cols = ['coal', 'lignite', 'gas', 'nuclear', 'hydro', 'wind', 'solar', 'other_re']
fuel_labels = ['Coal', 'Lignite', 'Gas', 'Nuclear', 'Hydro', 'Wind', 'Solar', 'Other RE']

annual = df.groupby('year')[fuel_cols + ['total', 'renewables']].sum() / 1000  # TWh

print('Annual Generation (TWh):')
print('=' * 80)
print(annual[fuel_cols + ['total']].round(0).to_string())
print()

# Growth rates
print('Year-over-Year Growth:')
for col in ['total', 'coal', 'solar', 'wind', 'hydro', 'nuclear', 'renewables']:
    g23 = (annual.loc[2023, col] / annual.loc[2022, col] - 1) * 100
    g24 = (annual.loc[2024, col] / annual.loc[2023, col] - 1) * 100
    g_2yr = (annual.loc[2024, col] / annual.loc[2022, col] - 1) * 100
    print(f'  {col.title():12s}: 2023 {g23:+5.1f}% | 2024 {g24:+5.1f}% | 2-year {g_2yr:+5.1f}%')

Annual Generation (TWh):
        coal  lignite   gas  nuclear  hydro  wind  solar  other_re   total
year                                                                      
2022  1127.0     37.0  26.0     46.0  164.0  66.0   86.0      14.0  1537.0
2023  1230.0     33.0  30.0     48.0  139.0  79.0  105.0       9.0  1680.0
2024  1292.0     34.0  34.0     54.0  143.0  79.0  125.0      10.0  1777.0

Year-over-Year Growth:
  Total       : 2023  +9.3% | 2024  +5.8% | 2-year +15.6%
  Coal        : 2023  +9.1% | 2024  +5.0% | 2-year +14.6%
  Solar       : 2023 +22.7% | 2024 +18.6% | 2-year +45.6%
  Wind        : 2023 +20.9% | 2024  -1.1% | 2-year +19.6%
  Hydro       : 2023 -15.0% | 2024  +2.9% | 2-year -12.5%
  Nuclear     : 2023  +4.5% | 2024 +13.3% | 2-year +18.4%
  Renewables  : 2023 +16.9% | 2024 +10.3% | 2-year +28.9%


## 2. Overlaid Daily Curves — Same Calendar, Different Years

In [3]:
# Key fuels to compare: coal, solar, wind, total
compare_fuels = [
    ('total', 'Total Generation'),
    ('coal', 'Coal'),
    ('solar', 'Solar'),
    ('wind', 'Wind'),
    ('hydro', 'Hydro'),
    ('nuclear', 'Nuclear'),
]

fig = make_subplots(rows=3, cols=2, subplot_titles=[t for _, t in compare_fuels],
                    vertical_spacing=0.1, horizontal_spacing=0.06)

for i, (col, title) in enumerate(compare_fuels):
    row, colnum = (i // 2) + 1, (i % 2) + 1
    for year in years:
        ydf = df[df['year'] == year].copy()
        smooth = ydf.set_index('day_of_year')[col].rolling(14, center=True).mean()
        fig.add_trace(go.Scatter(
            x=smooth.index, y=smooth,
            name=str(year), mode='lines',
            line=dict(width=2, color=year_colors[year]),
            showlegend=(i == 0),
            hovertemplate=f'{year}: %{{y:.0f}} MU<extra></extra>',
        ), row=row, col=colnum)
    fig.update_yaxes(title_text='MU/day', title_font_size=9, row=row, col=colnum)
    fig.update_xaxes(title_text='Day of Year', title_font_size=9, row=row, col=colnum)

fig.update_layout(
    title='Year-over-Year Comparison: 2022 vs 2023 vs 2024 (14-day avg)',
    height=800, plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
)
fig.show()

## 3. Solar Growth — The Fastest Mover

In [4]:
fig = go.Figure()

for year in years:
    ydf = df[df['year'] == year].copy()
    cum = ydf['solar'].cumsum() / 1000  # TWh
    fig.add_trace(go.Scatter(
        x=ydf['day_of_year'], y=cum.values,
        name=str(year), mode='lines',
        line=dict(width=2.5, color=year_colors[year]),
        hovertemplate=f'{year}: %{{y:.1f}} TWh<extra></extra>',
    ))

fig.update_layout(
    title='Cumulative Solar Generation Through the Year (TWh)',
    xaxis_title='Day of Year',
    yaxis_title='Cumulative Solar (TWh)',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=450,
)
fig.show()

for y in years:
    total_solar = df[df['year'] == y]['solar'].sum() / 1000
    print(f'  {y} total solar: {total_solar:.1f} TWh')

  2022 total solar: 85.8 TWh
  2023 total solar: 105.2 TWh
  2024 total solar: 124.9 TWh


## 4. The Big Question: Is Coal Declining?

In [5]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Absolute Coal Generation', 'Coal Share of Total (%)'])

for year in years:
    ydf = df[df['year'] == year].copy()
    
    # Absolute
    smooth = ydf.set_index('day_of_year')['coal'].rolling(14, center=True).mean()
    fig.add_trace(go.Scatter(
        x=smooth.index, y=smooth,
        name=str(year), mode='lines',
        line=dict(width=2, color=year_colors[year]),
        showlegend=True,
        hovertemplate=f'{year}: %{{y:.0f}} MU<extra></extra>',
    ), row=1, col=1)
    
    # Share
    coal_share = (ydf['coal'] / ydf['total'] * 100).rolling(14, center=True).mean()
    fig.add_trace(go.Scatter(
        x=ydf['day_of_year'].values, y=coal_share.values,
        name=str(year), mode='lines',
        line=dict(width=2, color=year_colors[year]),
        showlegend=False,
        hovertemplate=f'{year}: %{{y:.1f}}%<extra></extra>',
    ), row=1, col=2)

fig.update_yaxes(title_text='MU/day', row=1, col=1)
fig.update_yaxes(title_text='Coal %', row=1, col=2)
fig.update_xaxes(title_text='Day of Year', row=1, col=1)
fig.update_xaxes(title_text='Day of Year', row=1, col=2)

fig.update_layout(
    title='Coal: Absolute vs Share — Is It Really Declining?',
    height=450, plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
)
fig.show()

for y in years:
    coal_tot = df[df['year'] == y]['coal'].sum() / 1000
    coal_pct = df[df['year'] == y]['coal'].sum() / df[df['year'] == y]['total'].sum() * 100
    print(f'  {y}: {coal_tot:.0f} TWh coal ({coal_pct:.1f}% of total)')

  2022: 1127 TWh coal (73.4% of total)
  2023: 1230 TWh coal (73.2% of total)
  2024: 1292 TWh coal (72.7% of total)


## 5. Clean Energy Share Trend

In [6]:
fig = go.Figure()

for year in years:
    ydf = df[df['year'] == year].copy()
    smooth = ydf.set_index('day_of_year')['clean_share'].rolling(14, center=True).mean()
    fig.add_trace(go.Scatter(
        x=smooth.index, y=smooth,
        name=str(year), mode='lines',
        line=dict(width=2.5, color=year_colors[year]),
        hovertemplate=f'{year}: %{{y:.1f}}%<extra></extra>',
    ))

fig.update_layout(
    title='Clean Energy Share (%) — 2022 vs 2023 vs 2024 (14-day avg)',
    xaxis_title='Day of Year',
    yaxis_title='Clean Share (%)',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=450,
)
fig.show()

for y in years:
    cs = df[df['year'] == y]['clean_share'].mean()
    print(f'  {y} avg clean share: {cs:.1f}%')

  2022 avg clean share: 24.0%
  2023 avg clean share: 22.5%
  2024 avg clean share: 23.0%


## 6. Demand Growth vs Clean Growth — The Race

In [7]:
# Annual totals comparison
metrics = {
    'Total Demand': df.groupby('year')['total'].sum() / 1000,
    'Coal': df.groupby('year')['coal'].sum() / 1000,
    'All Clean': df.groupby('year').apply(lambda g: (g['nuclear'] + g['hydro'] + g['renewables']).sum()) / 1000,
    'Wind+Solar': df.groupby('year')['wind_solar'].sum() / 1000,
}

# Indexed to 2022 = 100
fig = go.Figure()
colors = {'Total Demand': '#2C3E50', 'Coal': '#922B21', 'All Clean': '#27AE60', 'Wind+Solar': '#F5B041'}

for name, series in metrics.items():
    indexed = series / series.iloc[0] * 100
    fig.add_trace(go.Scatter(
        x=[str(y) for y in years], y=indexed.values,
        name=name, mode='lines+markers',
        line=dict(width=3, color=colors[name]),
        marker=dict(size=10),
        hovertemplate=f'{name}: %{{y:.1f}} (index)<br>%{{text}} TWh<extra></extra>',
        text=[f'{v:.0f}' for v in series.values],
    ))

fig.add_hline(y=100, line_dash='dot', line_color='grey')

fig.update_layout(
    title='Growth Index (2022 = 100): Is Clean Energy Keeping Up With Demand?',
    yaxis_title='Index (2022 = 100)',
    plot_bgcolor='#FAFAF5', paper_bgcolor='#FAFAF5',
    height=450,
)
fig.show()

print('\nThe race:')
for name, series in metrics.items():
    g = (series.iloc[-1] / series.iloc[0] - 1) * 100
    print(f'  {name:15s}: {series.iloc[0]:.0f} → {series.iloc[-1]:.0f} TWh ({g:+.1f}% over 2 years)')


The race:
  Total Demand   : 1537 → 1777 TWh (+15.6% over 2 years)
  Coal           : 1127 → 1292 TWh (+14.6% over 2 years)
  All Clean      : 370 → 411 TWh (+11.0% over 2 years)
  Wind+Solar     : 151 → 203 TWh (+34.3% over 2 years)


## Key Findings

1. **Solar is the fastest-growing source** — quantified above. But from a small base.
2. **Coal in absolute terms** — still growing or flat? The answer defines the narrative.
3. **Coal share is slowly declining** even as absolute coal may rise — because demand is growing faster than coal.
4. **Clean energy is growing faster than demand** — but the gap between the two is still vast.
5. **The tension is real:** India needs both more renewables AND more coal to meet demand growth. This isn't a simple transition story.
6. **Wind+Solar growth is the most promising signal** — growing faster than total demand.

→ Next: Notebook 07 — Notable Events